# Apuntes completos - Notebook 06: PyTorch Transfer Learning

Este cuaderno resume de punta a punta el notebook 06 sobre transfer learning en vision con PyTorch.

Objetivo practico: entrenar un clasificador de imagen (pizza, steak, sushi) reutilizando un modelo preentrenado para ganar precision y velocidad.

## 1. Idea central

Transfer learning consiste en tomar un modelo entrenado en un dataset grande (ej. ImageNet) y adaptarlo a tu problema.

Dos estrategias:
1. Feature extraction: congelar capas base y entrenar solo la cabeza final (classifier).
2. Fine-tuning: descongelar parte o todo el modelo y ajustar con learning rate bajo.

En este notebook se usa sobre todo feature extraction.

## 2. Fuentes de modelos preentrenados

- torchvision.models
- timm
- Hugging Face Hub

El notebook trabaja con EfficientNet_B0 desde torchvision.

In [ ]:
# Setup base (versiones e imports clave)
import torch
import torchvision

assert int(torch.__version__.split('.')[1]) >= 12
assert int(torchvision.__version__.split('.')[1]) >= 13

from torch import nn
from torchvision import transforms
from torchinfo import summary

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

## 3. Dataset usado

Se usa FoodVision Mini (pizza/steak/sushi), tipicamente con estructura:

- data/pizza_steak_sushi/train
- data/pizza_steak_sushi/test

Clase final: 3 etiquetas -> ['pizza', 'steak', 'sushi'].

## 4. Transformaciones de imagen

Opcion A (manual): Resize + ToTensor + Normalize (stats de ImageNet).

Opcion B (recomendada): usar transforms automaticas del peso preentrenado.

Ventaja de la opcion B: garantizas que el preprocesado coincide con el del entrenamiento original del modelo.

In [ ]:
# API moderna de pesos (torchvision >= 0.13)
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
auto_transforms = weights.transforms()

manual_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

auto_transforms

## 5. DataLoaders

Se crean DataLoaders para train y test (batch_size=32) usando los transforms definidos.

Salida esperada:
- train_dataloader
- test_dataloader
- class_names

In [ ]:
# Ejemplo de creacion (si usas el modulo going_modular del repo)
from pathlib import Path
from going_modular.going_modular import data_setup

train_dir = Path('data/pizza_steak_sushi/train')
test_dir = Path('data/pizza_steak_sushi/test')

train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=auto_transforms,
    batch_size=32
)

class_names

## 6. Modelo preentrenado y personalizacion

Se carga EfficientNet_B0 con pesos preentrenados y luego:
1. Se congelan las capas de features (base).
2. Se reemplaza classifier para 3 clases.

Esto reduce mucho los parametros entrenables y acelera entrenamiento.

In [ ]:
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
model = torchvision.models.efficientnet_b0(weights=weights).to(device)

# Congelar extractor de caracteristicas
for param in model.features.parameters():
    param.requires_grad = False

# Reemplazar cabezal de clasificacion
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=1280, out_features=3, bias=True)
).to(device)

summary(model=model, input_size=(32, 3, 224, 224),
        col_names=['input_size', 'output_size', 'num_params', 'trainable'])

## 7. Entrenamiento

Configuracion principal usada:
- Loss: CrossEntropyLoss
- Optimizador: Adam
- Learning rate: 0.001
- Epochs: 5

Con extractor congelado, el entrenamiento suele ser rapido y estable.

In [ ]:
from going_modular.going_modular import engine

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

results = engine.train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=5,
    device=device
)

results.keys()

## 8. Evaluacion

Se grafican curvas de train/test loss y accuracy para verificar:
- convergencia
- estabilidad
- posible overfitting

En el notebook, EfficientNet_B0 mejora claramente frente a TinyVGG.

In [ ]:
from helper_functions import plot_loss_curves
plot_loss_curves(results)

## 9. Inferencia

Patron correcto para predecir:
1. model.eval()
2. with torch.inference_mode()
3. logits -> softmax -> argmax

No olvidar aplicar el mismo transform que en entrenamiento.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def predict_image(model, image_path, class_names, transform, device):
    model.eval()
    img = Image.open(image_path).convert('RGB')
    x = transform(img).unsqueeze(0).to(device)

    with torch.inference_mode():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)

    pred_idx = int(torch.argmax(probs, dim=1).item())
    conf = float(torch.max(probs).item())

    plt.imshow(img)
    plt.title(f'Pred: {class_names[pred_idx]} | Prob: {conf:.3f}')
    plt.axis('off')

    return class_names[pred_idx], conf

## 10. Hiperparametros del notebook

- Arquitectura: EfficientNet_B0
- Pesos: EfficientNet_B0_Weights.DEFAULT
- Input size: 224x224
- Batch size: 32
- Epochs: 5
- Optimizer: Adam
- LR: 0.001
- Dropout: 0.2
- Clases: 3

## 11. Resultados y lectura

Mensajes clave del notebook:
- El salto de rendimiento frente a modelos pequenos entrenados desde cero es grande.
- Se obtiene mejor accuracy con menos tiempo de entrenamiento.
- Transfer learning funciona muy bien con pocos datos cuando el dominio es parecido.

## 12. Checklist de buenas practicas

- Verificar versiones de torch y torchvision.
- Usar API nueva de weights, no pretrained=True.
- Mantener consistencia de transforms entre train, test e inferencia.
- Revisar requires_grad para confirmar que solo entrenas lo esperado.
- Usar device-agnostic code.
- Para inferencia: eval + inference_mode.
- Si haces fine-tuning real, bajar learning rate.

## 13. Diferencia rapida: Feature extraction vs Fine-tuning

Feature extraction:
- Congela base
- Entrena cabeza
- Rapido y estable
- Ideal para pocos datos

Fine-tuning:
- Descongela parte o todo
- Ajuste mas fino
- Mejor potencial de accuracy
- Requiere mas cuidado (lr bajo, mas compute, mas datos)

## 14. Conclusiones finales

El notebook 06 deja una receta muy util en proyectos reales:
1. elegir arquitectura preentrenada
2. adaptar transforms
3. congelar base y reemplazar classifier
4. entrenar rapido
5. medir y visualizar
6. hacer inferencia consistente

Si tu baseline desde cero rinde poco, transfer learning suele ser la primera mejora de alto impacto.